# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [2]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [3]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Sat Nov 21 01:14:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [4]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [5]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 21 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 21 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 21 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [6]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [7]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 What are the benefits and drawbacks of this type of computing? What are the main differences between quantum computing and classical computing? What are the most important advantages and disadvantages of quantum computing? Why quantum computing is important? Why is quantum computing important in the future? How does quantum computing work? Is quantum computing possible? Why is quantum computing so important? How quantum computing is different? What are the applications of quantum computing? What is quantum computing used for? Is quantum computing a good thing? How is quantum computing different from classical computing?
Explain me, what is quantum computing?
What are the benefits and drawbacks of this type of computing?
What are the main differences between quantum computing and classical computing?
What are the most important advantages and disadvantages of quantum computing

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a type of computin

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [8]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve 15 × 24, we can use the standard multiplication algorithm. Here's how we can break it down:

First, let's write the numbers vertically, aligning the digits by place value:

```
  15
× 24
------
```

Now, we'll multiply 15 by the units digit of 24, which is 4:

```
  15
×  4
------
  60
```

Next, we'll multiply 15 by the tens digit of 24, which is 2, but we need to shift the result one place to the left (since it's in the tens place):

```
  15
×  2
------
  30
```

Now, we'll add the two results together:

```
  60
+ 30
------
  90
```

So, 15 × 24 = 90.

Alternatively, we can also use the distributive property to break down the multiplication:

15 × 24 = 15 × (20 + 4) = 15 × 20 + 15 × 4 = 300 + 60 = 360.

Wait, that's not correct. Let's correct that:

15 × 24 = 15 × (20 + 4) = 15 × 20 + 15 × 4 = 300 + 60 = 360.

Wait, no, that's still incorrect. Let

------------------------------------------

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 21 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [10]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [11]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [12]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [13]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [14]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [15]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 21 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## Відповіді на теоретичні питання

### Питання 11. Як довго виконувалися клітинки ноутбуку 02_chat_templates? Чи були проблеми з запуском, які?

Ноутбук запускався в Google Colab на GPU Tesla T4. Тренування в ньому немає, тож час витрачається на завантаження двох моделей по 3 млрд параметрів (`SmolLM3-3B-Base` і `SmolLM3-3B`, у fp16 по ≈6 ГБ кожна) і на генерацію: до 150 токенів у порівнянні Base/Instruct і до 300 токенів у задачах на міркування (режим `/think` найдовший).

Загальний час `Run all` — орієнтовно **≈10–15 хв** (більшість часу йде на завантаження двох моделей).

Помилок не було, усе виконалося з першого запуску. З'являлися лише інформаційні повідомлення: попередження про відсутність `HF_TOKEN`, попередження про `clean_up_tokenization_spaces` для BPE-токенізатора та про параметри RoPE у конфігурації Qwen3 (на результат не впливають). Окремо варто відзначити, що ваги SmolLM3 збережені в bfloat16, а T4 його апаратно не підтримує, тому в коді моделі завантажуються у float16. Без цього були б повільні обчислення або помилки.

### Питання 12. Для чого змінювати параметр `add_generation_prompt` у `apply_chat_template()`? Як за згенерованою розмовою дізнатися його значення?

- `add_generation_prompt=False` — діалог **завершений**: рядок закінчується останнім повідомленням (у виводі секції 2 це `<|im_end|>`). Так форматують **тренувальні дані** (SFT), де відповідь асистента вже є.
- `add_generation_prompt=True` — у кінець додається **заголовок відповіді асистента** `<|im_start|>assistant`, щоб модель знала, що тепер її черга, і почала генерувати відповідь. Так форматують запит для **інференсу**.

Якщо поставити `True` для тренувальних даних, у кінці з'явиться зайвий порожній заголовок асистента; якщо поставити `False` для інференсу, модель не отримає сигналу про початок своєї відповіді й може продовжити текст користувача.

Як визначити значення за готовим текстом (видно в секції 2): якщо рядок закінчується **відкритим** заголовком `<|im_start|>assistant` без вмісту й без `<|im_end|>`, то було `True`. У режимі `/no_think` після заголовка стоїть ще й порожній блок `<think></think>`. Якщо останнє повідомлення закрито `<|im_end|>`, то було `False`.

### Питання 13. Чим відрізняються ролі `system` і `assistant` у шаблонах розмов?

- **`system`** — інструкції для моделі від розробника: роль, стиль, обмеження, прапорці режиму (`/think`, `/no_think`). Стоїть на початку діалогу, зазвичай один раз; модель його **не генерує**, а лише читає як контекст.
- **`assistant`** — відповіді самої моделі (або приклади таких відповідей у тренувальних даних чи в історії діалогу). У режимі інференсу модель генерує саме вміст після заголовка `assistant`.

Різниця є й у токенах (`<|im_start|>system` проти `<|im_start|>assistant`), і в тренуванні: під час SFT loss зазвичай рахують лише на токенах `assistant`
(`assistant_only_loss=True`), а system і user слугують умовою, а не ціллю навчання.

### Питання 14. Прокоментуйте вивід даних у секції 2 «Як шаблон розмови форматує діалог». Які секції було додано шаблонізатором і для чого?

У вхідних даних були лише повідомлення з ролями, а шаблон SmolLM3 перетворив їх на довший рядок і сам додав:
1. **Службові токени ChatML**: кожне повідомлення обгорнуте в `<|im_start|>роль … <|im_end|>`. Вони позначають межі та авторів повідомлень.
2. **Блок `## Metadata`** у системному повідомленні: дата відсічення знань (June 2025), поточна дата (21 November 2026, шаблон підставляє її під час виклику) і `Reasoning Mode: /think` або `/no_think`. Саме цей режим ми задали прапорцем у тексті `system`; сам прапорець із тексту інструкцій шаблон прибрав.
3. **Блок `## Custom Instructions`** — наш системний текст. Якщо власної інструкції немає (приклад `simple_qa`), шаблон підставляє типову («You are a helpful AI assistant named SmolLM…»). Для `/think` без власного тексту (`reasoning_task`) підставляється довга інструкція про структуру «Thought / Solution».
4. **Порожній блок `<think>\n\n</think>`** у режимі `/no_think`: у варіанті для інференсу він стоїть після `<|im_start|>assistant`, а в багатоходовому діалозі — і в попередніх відповідях асистента. Так модель отримує сигнал не міркувати. У режимі `/think` рядок для інференсу закінчується на `<|im_start|>assistant`, і блок міркувань модель генерує сама.
5. Різниця між варіантами: з `add_generation_prompt=True` додається заголовок асистента (див. питання 12).

Тобто модель бачить значно більше, ніж ми їй передали, і саме в такому вигляді потрібно готувати дані для донавчання.

### Питання 15. Прокоментуйте різницю між Base і Instruct, проглянувши вивід у секції 3.

Обидві моделі отримали один і той самий запит «Explain me, what is quantum computing?».
- **Base** (без шаблону, сирий текст) **не відповіла на питання**, а продовжила текст, як продовжують веб-сторінку: дописала до запиту нове питання («What does it mean for the future?»), а далі написала популярну статтю з вступом про наукову фантастику й квантову суперпозицію, обірвавши на ліміті токенів. Base-модель навчена лише передбачати наступний токен у великих корпусах текстів і не знає, що від неї чекають відповіді.
- **Instruct** (діалог через шаблон із `/no_think`) одразу дала пряму структуровану відповідь («Quantum computing is a revolutionary field… qubits…»). Вона додатково донавчена на діалогах (SFT та вирівнювання) і вміє виконувати інструкції та розпізнає ролі шаблону.

Висновок: Base — сира «мовна» модель, яка підходить як основа для донавчання, а Instruct придатна для безпосереднього спілкування. Щоб змусити Base відповідати, потрібні few-shot приклади в промпті або власне SFT.

### Питання 16. Чому в режимі `/think` при генерації було передано `max_new_tokens=300`?

У режимі `/think` модель спочатку генерує **ланцюжок міркувань** у блоці `<think>…</think>` і лише потім відповідь, тож текст значно довший, ніж у `/no_think`. Ліміт 300 токенів — компроміс для демо: він обмежує час генерації й памʼять GPU.

Вивід показує, що для `/think` **цього не вистачило**: усі три відповіді обірвалися посередині міркувань, до фінальної відповіді модель не дійшла (наприклад, у задачі 3 розрахунок зупинився на відніманні $31.05 від $50). У режимі `/no_think` 300 токенів було достатньо, відповіді завершені. Тому для `/think` потрібен значно більший ліміт (порядку 1000–2000 токенів), інакше відповідь доводиться вгадувати з незакінчених міркувань.

### Питання 17. Чи впоралася модель з математичними задачами? Який режим генерації Ви обрали б для себе?

- **`/no_think`:** усі три задачі розв'язано правильно і з поясненням кроків: 15 × 24 = 360; для 30 печив потрібно 5 склянок борошна; залишок 50 − (18.75 + 12.30) = **$18.95**.
- **`/think`:** фінальних відповідей ми не побачили, бо генерація обірвалась на ліміті в 300 токенів (див. питання 16). Видно, що міркування були правильними (проміжні 120 і 31.05 коректні) і що модель багато разів перевіряє себе («Wait, that seems like a lot, but let me check my math again»).

Для таких простих арифметичних задач я обрав би **`/no_think`**: він дає правильну відповідь значно швидше та з меншою кількістю токенів. Режим `/think` має сенс для складніших задач (багатокрокова логіка, доведення, код), де перевірка власних кроків реально підвищує точність. Він коштує в рази більше токенів і часу. З цього експерименту не можна зробити висновок, що `/think` точніший, бо до відповіді він не дійшов.

### Питання 18. Які відмінності в шаблонах розмов Ви помітили в секції 5?

Один і той самий діалог (`system`: «You are a helpful assistant.», `user`: «Hi!», `assistant`: «Hello! How can I help?») три шаблони відформатували по-різному:
- **SmolLM3-3B:** ChatML-токени `<|im_start|>роль … <|im_end|>`, а також сам додає блок `## Metadata` (дати, `Reasoning Mode: /think` за замовчуванням) і розділ `## Custom Instructions` із нашим системним текстом.
- **Qwen3-0.6B:** ті самі ChatML-токени, але без метаданих: системне повідомлення вставлено дослівно. Зате у відповідь асистента автоматично доданий порожній блок `<think>\n\n</think>`.
- **Phi-3.5-mini:** інші спеціальні токени: `<|system|>`, `<|user|>`, `<|assistant|>`, кожне повідомлення закривається `<|end|>`, а в кінці стоїть ще `<|endoftext|>`.

Прапорець режиму міркування теж задається по-різному: у SmolLM3 це текст `/think` або `/no_think` у системному повідомленні, а в Qwen3 — параметр `enable_thinking` в `apply_chat_template`. При `True` рядок для інференсу закінчується на `<|im_start|>assistant`, а при `False` — ще й порожнім блоком `<think></think>`. Звідси висновок: шаблон і токенізатор треба брати саме від тієї моделі, яку використовуємо чи донавчаємо.

### Питання 19. Для чого знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?

Модель бачить не «ролі», а послідовність токенів, і саме шаблон розмови перетворює `messages` на цю послідовність. Тому:
1. **Формат тренування має збігатися з форматом інференсу.** Якщо донавчати з одним шаблоном, а запитувати з іншим (або без `add_generation_prompt`), модель отримає непередбачуваний вхід, і якість впаде.
2. **Кожна модель має свій шаблон** (ChatML у SmolLM3 і Qwen3, інший у Phi-3.5): дані треба форматувати шаблоном саме тієї моделі, яку донавчаємо.
3. **Base-моделі шаблону зазвичай не мають або не бачили його токенів**, тож його треба задати вручну (наприклад, взяти з instruct-версії).
4. **Шаблон додає службові токени й метадані** (дата, режим міркування, системні інструкції за замовчуванням), які модель теж бачить під час тренування. Режим міркування в даних має узгоджуватися з тим, як модель використовуватимуть.
5. **Маскування loss** (`assistant_only_loss`) спирається на розмітку шаблону: треба знати, де починається й закінчується відповідь асистента.

### Питання 20. З яких даних було побудовано датасет `HuggingFaceTB/smoltalk2`?

SmolTalk2 — відкритий набір даних, на якому проходило постнавчання моделі SmolLM3-3B. Конфігурація SFT складається з **25 частин** (вивід клітинки з `get_dataset_split_names`). За змістом вони поділяються так:
- **Міркування (`think`)**: `OpenThoughts3_1.2M`, `s1k_1.1`, `multi_turn_reasoning_if`; у `no_think`-версіях — `Mixture_of_Thoughts_science` і `OpenThoughts3_1.2M`.
- **Загальні діалоги та інструкції**: `OpenHermes_2.5`, `tulu_3_sft_personas_instruction_following`, набір `smoltalk_smollm3_*` (everyday_conversations, explore_instruct_rewriting, smol_magpie_ultra, smol_rewrite, smol_summarize, systemchats_30k) і `smoltalk_systemchats`.
- **Багатомовні дані**: `aya_dataset`, `smoltalk_multilingual8` і `smoltalk_multilingual_8languages`.
- **Довгий контекст**: `LongAlign_64k` (дві версії).
- **Таблиці**: `table_gpt` (дві версії).
- **Виклик інструментів і агенти**: `hermes_function_calling_v1`, `xlam_traces`, `smolagents_toolcalling_traces`.

Суфікс `think` або `no_think` показує, чи містять відповіді блок міркувань. Багато частин мають обидві версії, щоб модель вміла працювати в обох режимах. Частини з `Qwen3_32B` у назві — **синтетичні дані**, відповіді в яких згенерувала модель Qwen3-32B (так само й `smoltalk_everyday_convs_reasoning_Qwen3_32B_think`, яку ми завантажували). Решта — наявні відкриті датасети. За описом датасету, окрім SFT, у ньому є ще підмножини Mid (проміжне навчання на міркуваннях) і Preference (пари відповідей для вирівнювання).

### Питання 21. Які модифікації даних відбуваються в секції «Перетворення власного датасету у формат `messages`»?

Вихідний GSM8K має колонки `question` і `answer`. Функція `to_messages` разом із `map(batched=True)` робить таке:
1. Кожну пару «питання — відповідь» перетворює на **список із трьох повідомлень з ролями**: `system` (однаковий для всіх прикладів: «You are a math tutor. Solve problems step by step.»), `user` (питання), `assistant` (еталонний розв'язок із GSM8K без змін).
2. Через `remove_columns=gsm8k.column_names` видаляє старі колонки `question` і `answer`, тож у датасеті лишається одна колонка `messages`, яку очікує `SFTTrainer`.
3. Для демо береться лише 100 перших прикладів (`train[:100]`).

Сам текст відповіді не змінюється, змінюється **структура**: датасет приводиться до формату `messages`, до якого вже можна застосувати шаблон розмови.